# 28 · Tool errors the model can recover from
Level L3 · Part 4 · core · ~30 min · builds on 26

## Goal
Change lesson 26's loop so a failing tool no longer crashes the run. The failure goes back to the model
as an **error result** with a message that says how to fix the call. Then watch Scout recover: its
`write_file` refuses a path outside Scout's folder, and the model retries with a valid name.

## The idea
Tools fail: bad arguments, a missing file, a path the tool must refuse. In lesson 26 any exception
escaped `run_agent` and the whole run died. But the model caused the failure, so the model is the one
who can fix it. We catch the exception and send it back as the tool's result, marked as an error. This
is **error as observation**: the error is just one more thing the model sees. A useful message names
what was wrong and what to do instead. Think of a form on a website: "invalid input" makes you guess,
"date must look like 2026-10-08" lets you fix it on the next try.

## Picture
```mermaid
flowchart TD
    C[model asks for a tool] --> R{run the tool}
    R -- returns --> OK[result: is_error false]
    R -- raises --> E[result: is_error true + how to fix it]
    OK --> H[append to history]
    E --> H
    H --> N[next model call: it reads the error and retries]
```

## Step by step
### 1. Start the recorder and set up
The recorder first. Without a key the Gemini calls replay from `../data/llm_cassettes/28_tool-errors.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("28_tool-errors")

LLM mode: replay (28_tool-errors.yaml)


Setup from lesson 26, unchanged: the notes, the client, the pinned model, `search_notes` and `calculator`.

In [2]:
import ast, json, operator, os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]

The safe calculator from lesson 26, unchanged.

In [3]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def calculator(expression: str) -> float:
    """Evaluate +, -, *, / on numbers. Nothing else is allowed."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        raise ValueError(f"not allowed in a calculation: {ast.dump(node)[:40]}")
    return ev(ast.parse(expression, mode="eval").body)

And lesson 26's `ScriptedModel` with its history check, unchanged.

In [4]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

### 2. A `write_file` that refuses bad paths
Scout gets a `write_file` tool. This one checks the path properly: a name with a folder in it, like `../reports/mars.txt`, could write anywhere on your disk, so the tool
raises. The message says what was wrong and gives a name that would work.

In [5]:
SCOUT_DIR = Path("../data/scout_files")
SCOUT_DIR.mkdir(parents=True, exist_ok=True)
for old in SCOUT_DIR.glob("*.txt"):                         # start each run with an empty folder
    old.unlink()

def write_file(filename: str, text: str) -> str:
    """Save text to a file in Scout's folder. Only plain file names are allowed."""
    if not filename or Path(filename).name != filename or filename in (".", ".."):
        raise ValueError(f"'{filename}' is not a plain file name: write_file only saves inside "
                         f"Scout's folder. Retry with just a name, e.g. '{Path(filename).name or 'notes.txt'}'.")
    (SCOUT_DIR / filename).write_text(text)
    return f"saved {len(text)} characters to {filename}"

print(write_file("mars_day.txt", "A day on Mars lasts about 1479 minutes."))
print(sorted(p.name for p in SCOUT_DIR.iterdir()))

saved 39 characters to mars_day.txt
['mars_day.txt']


The schema. Its `filename` description is loose on purpose ("name of the file"), so a model may well
send a path. Lesson 29 is about writing schemas that prevent this up front.

In [6]:
WRITE_FILE_SCHEMA = {
    "name": "write_file",
    "description": "Save text to a file in Scout's folder, replacing it if it exists.",
    "parameters": {"type": "object", "required": ["filename", "text"],
                   "properties": {"filename": {"type": "string", "description": "name of the file"},
                                  "text": {"type": "string", "description": "the full text to save"}}},
}
SEARCH_NOTES_SCHEMA = {
    "name": "search_notes",
    "description": "Search the user's notes. Returns every note whose text contains the query word "
                   "(case-insensitive), as a list of {id, text}.",
    "parameters": {"type": "object", "required": ["query"],
                   "properties": {"query": {"type": "string", "description": "one word, e.g. 'Mars'"}}},
}
TOOLS = {"search_notes": search_notes, "calculator": calculator, "write_file": write_file}
SCHEMAS = [SEARCH_NOTES_SCHEMA, WRITE_FILE_SCHEMA]           # the Gemini run below needs only these two
print("tools:", list(TOOLS), "| schemas sent to the model:", [s["name"] for s in SCHEMAS])

tools: ['search_notes', 'calculator', 'write_file'] | schemas sent to the model: ['search_notes', 'write_file']


### 3. Lesson 26's loop crashes
Here is lesson 26's `run_agent`, unchanged. The fake model asks to write to `../reports/mars.txt`.
The tool raises inside the loop, and the exception ends the run. The model never learns what happened.

In [7]:
def run_agent_26(question, model, tools, schemas, max_steps=6):
    messages = [{"role": "user", "text": question}]
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        if not reply.get("tool_calls"):
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step, "messages": messages}
        for call in reply["tool_calls"]:
            result = tools[call["name"]](**call["args"])
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"],
                             "content": json.dumps(result)})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "messages": messages}

BAD_PATH = {"tool_calls": [{"name": "write_file",
                            "args": {"filename": "../reports/mars.txt", "text": "Mars notes"}}]}
try:
    run_agent_26("Save 'Mars notes' to ../reports/mars.txt", ScriptedModel([BAD_PATH]), TOOLS, SCHEMAS)
except ValueError as e:
    print("run crashed:", e)

run crashed: '../reports/mars.txt' is not a plain file name: write_file only saves inside Scout's folder. Retry with just a name, e.g. 'mars.txt'.


### 4. Run a tool, never raise
`run_tool` runs one tool call and always returns a result. On success it is the JSON of the return
value with `is_error` False. On any exception it is a short message, the exception's type and text,
with **`is_error`** True. An unknown tool name is an error result too, listing the real tools. No stack
trace: it wastes tokens and can leak file paths.

In [8]:
def run_tool(call: dict, tools: dict) -> dict:
    """Run one tool call. Returns {"content": str, "is_error": bool}; never raises."""
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available tools: {', '.join(tools)}.",
                "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:                                   # any failure becomes an observation
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

for call in [{"name": "calculator", "args": {"expression": "24*60+39"}},
             {"name": "write_file", "args": {"filename": "../reports/mars.txt", "text": "x"}},
             {"name": "save_file", "args": {}},
             {"name": "calculator", "args": {"expr": "1+1"}}]:
    print(run_tool(call, TOOLS))

{'content': '1479', 'is_error': False}
{'content': "ValueError: '../reports/mars.txt' is not a plain file name: write_file only saves inside Scout's folder. Retry with just a name, e.g. 'mars.txt'.", 'is_error': True}
{'content': "unknown tool 'save_file'. Available tools: search_notes, calculator, write_file.", 'is_error': True}
{'content': "TypeError: calculator() got an unexpected keyword argument 'expr'", 'is_error': True}


Four calls, one success and three error results, no crash. The last one is a wrong argument name:
Python's `TypeError` text ("unexpected keyword argument 'expr'") is already enough for a model to fix it.

### 5. The loop with error results
One change to lesson 26's loop: it calls `run_tool` and copies `is_error` into the tool message. It
also counts the errors, so the caller can see how bumpy the run was.

In [9]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 6) -> dict:
    messages, errors = [{"role": "user", "text": question}], 0
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            print(f"step {step}: answer")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step,
                    "errors": errors, "messages": messages}
        for call in calls:
            result = run_tool(call, tools)                    # never raises
            errors += result["is_error"]
            print(f"step {step}: {call['name']}({call['args']}) -> "
                  f"{'ERROR ' if result['is_error'] else ''}{result['content'][:70]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors, "messages": messages}

### 6. A fake model that reads the error
A real model reads the error text and adjusts. Our fake imitates that with one rule: if the last result
is an error that suggests a name (`e.g. '...'`), retry with it; otherwise retry the same call.

In [10]:
import re

def retry_from_error(messages: list[dict]) -> dict:
    last = messages[-1]
    if not last.get("is_error"):
        return {"text": f"Done: {json.loads(last['content'])}."}
    hint = re.search(r"e\.g\. '([^']+)'", last["content"])
    old_args = messages[-2]["tool_calls"][0]["args"]
    new_name = hint.group(1) if hint else old_args["filename"]
    return {"tool_calls": [{"name": "write_file", "args": {**old_args, "filename": new_name}}]}

SAVE_SCRIPT = [BAD_PATH, retry_from_error, retry_from_error]
run = run_agent("Save 'Mars notes' to ../reports/mars.txt", ScriptedModel(SAVE_SCRIPT), TOOLS, SCHEMAS)
print(run["stop"], "| errors:", run["errors"], "|", run["answer"])

step 1: write_file({'filename': '../reports/mars.txt', 'text': 'Mars notes'}) -> ERROR ValueError: '../reports/mars.txt' is not a plain file name: write_file
step 2: write_file({'filename': 'mars.txt', 'text': 'Mars notes'}) -> "saved 10 characters to mars.txt"
step 3: answer
end_of_turn | errors: 1 | Done: saved 10 characters to mars.txt.


Step 1 failed, step 2 used the suggested name and worked, step 3 answered. The run recovered on its own.

### 7. Why the message matters
Same fake, same loop, but a `write_file` that only says "permission denied". The fake has nothing to
go on, retries the same path, and burns every step.

In [11]:
def write_file_vague(filename: str, text: str) -> str:
    if Path(filename).name != filename:
        raise PermissionError("permission denied")
    return write_file(filename, text)

vague_tools = {**TOOLS, "write_file": write_file_vague}
run = run_agent("Save 'Mars notes' to ../reports/mars.txt", ScriptedModel(SAVE_SCRIPT), vague_tools,
                SCHEMAS, max_steps=3)
print(run["stop"], "| errors:", run["errors"])

step 1: write_file({'filename': '../reports/mars.txt', 'text': 'Mars notes'}) -> ERROR PermissionError: permission denied
step 2: write_file({'filename': '../reports/mars.txt', 'text': 'Mars notes'}) -> ERROR PermissionError: permission denied
step 3: write_file({'filename': '../reports/mars.txt', 'text': 'Mars notes'}) -> ERROR PermissionError: permission denied
step_limit | errors: 3


A real model is smarter than this fake, but the lesson holds: an **actionable error message** says what
was wrong *and* what a valid call looks like. "Permission denied" leaves the model guessing.

### 8. How the API marks an error (for reference)
Gemini's `FunctionResponse.response` takes an `"error"` key instead of `"output"` (google-genai field
docs), which step 9 uses; Anthropic's `tool_result` block has an `is_error` field instead.

In [12]:
err = run_tool(BAD_PATH["tool_calls"][0], TOOLS)
err["content"] = err["content"][:28] + "..."                  # shortened so each line fits
anthropic_block = {"type": "tool_result", "tool_use_id": "toolu_01", "content": err["content"],
                   "is_error": err["is_error"]}
gemini_response = {"error": err["content"]} if err["is_error"] else {"output": json.loads(err["content"])}
for name, item in [("anthropic", anthropic_block), ("gemini", gemini_response)]:
    print(f"{name:9}", json.dumps(item))

anthropic {"type": "tool_result", "tool_use_id": "toolu_01", "content": "ValueError: '../reports/mars...", "is_error": true}
gemini    {"error": "ValueError: '../reports/mars..."}


### 9. Gemini behind the same interface
Lesson 26's `to_contents`, with one change: a tool result goes into `response` under `"error"` or
`"output"`, following its `is_error` flag. The model's own turns still go back unchanged (thought
signatures, lesson 22).

In [13]:
def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

`GeminiModel` is lesson 26's, unchanged. The system prompt adds one sentence about tool errors.

In [14]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "If a tool returns an error, read it, fix the call and try again. Answer in one sentence.")

class GeminiModel:
    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.FunctionDeclaration(name=t["name"], description=t["description"],
                                           parameters_json_schema=t["parameters"]) for t in tools]
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=400,
                                          tools=[types.Tool(function_declarations=decls)],
                                          automatic_function_calling=NO_AFC)
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text, "raw": raw}

### 10. Watch Gemini recover
We ask for a path outside Scout's folder. Gemini picks the arguments itself; the loop only reports.

In [15]:
ASK = "Find my note about the length of a day on Mars and save its text to ../reports/mars_day.txt."
run = run_agent(ASK, GeminiModel(), TOOLS, SCHEMAS)
print(run["stop"], "| errors:", run["errors"], "|", run["answer"])
print("Scout's folder:", sorted(p.name for p in SCOUT_DIR.iterdir()))
for m in run["messages"]:                                     # the full text of each error the model read
    if m["role"] == "tool" and m["is_error"]:
        print("error the model read:", m["content"])

step 1: search_notes({'query': 'Mars'}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
step 2: write_file({'text': 'A day on Mars lasts about 24 hours and 39 minutes.', 'filename': '../reports/mars_day.txt'}) -> ERROR ValueError: '../reports/mars_day.txt' is not a plain file name: write_
step 3: write_file({'text': 'A day on Mars lasts about 24 hours and 39 minutes.', 'filename': 'mars_day.txt'}) -> "saved 50 characters to mars_day.txt"
step 4: answer
end_of_turn | errors: 1 | I have found your note about the length of a day on Mars and saved its text to mars_day.txt.
Scout's folder: ['mars.txt', 'mars_day.txt']
error the model read: ValueError: '../reports/mars_day.txt' is not a plain file name: write_file only saves inside Scout's folder. Retry with just a name, e.g. 'mars_day.txt'.


Read the trace above: the `write_file` call with the `../reports/` path came back as an ERROR. The
trace cuts results at 70 characters, so the last line prints the whole message, which ends "Retry with
just a name, e.g. 'mars_day.txt'." The next call used exactly that plain name. Nothing outside Scout's folder was written, and
the run ended normally with `end_of_turn`. (`mars.txt` in the folder is from the fake run in step 6.)

## At work
- **Error results are the standard pattern.** Anthropic's tool docs set `is_error: true` on the
  `tool_result`, and note that Claude retries an invalid tool call 2 or 3 times before apologising.
  Good error messages are part of tool design. → lesson 29
- **Error-recovery is unit-tested with a scripted fake**, like `SAVE_SCRIPT` here: a tool fails, the
  next call must carry the error result, the run must still end. → lesson 46
- **Least-privilege tools.** A file tool is confined to one folder and refuses anything else, as
  `write_file` does; tool results are also treated as untrusted input. → lesson 51
- **Error counts are logged per run.** A tool whose error rate is climbing usually means a bad schema
  or error message; it shows up in the logs before users complain. → lesson 49

## What just happened
- Lesson 26's loop died on the first tool exception, and the model never saw it.
- `run_tool` turns every outcome into a result: success with `is_error` False, or an exception's type
  and message with `is_error` True. Unknown tools and wrong arguments become error results too.
- The new `run_agent` appends that result like any other: **error as observation**.
- With an actionable message the fake model recovered in one retry; with "permission denied" it hit the
  step limit.
- Gemini received the error under the `"error"` key, corrected the path itself, and finished.

## Common mistakes
**1. Catching only the errors you expect.** This `run_tool` catches `ValueError` from `write_file`. Then
the model sends a wrong argument name, Python raises `TypeError`, and the run crashes after all.

In [16]:
def run_tool_narrow(call, tools):
    try:
        return {"content": json.dumps(tools[call["name"]](**call["args"])), "is_error": False}
    except ValueError as e:
        return {"content": f"ValueError: {e}", "is_error": True}

run_tool_narrow({"name": "calculator", "args": {"expr": "24*60+39"}}, TOOLS)

TypeError: calculator() got an unexpected keyword argument 'expr'

Fix: catch `Exception`, as `run_tool` does. Every tool failure belongs to the model. But send back only
the exception's type and text: never a traceback, a full file path or a secret from the environment.

In [17]:
print(run_tool({"name": "calculator", "args": {"expr": "24*60+39"}}, TOOLS))

{'content': "TypeError: calculator() got an unexpected keyword argument 'expr'", 'is_error': True}


**2. Looking up the tool outside the `try`.** Models sometimes call a tool that does not exist. Here the
dictionary lookup runs before the `try`, so the made-up name `save_file` raises `KeyError`.

In [18]:
def run_tool_lookup_outside(call, tools):
    fn = tools[call["name"]]                                 # KeyError for a made-up name
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

run_tool_lookup_outside({"name": "save_file", "args": {"filename": "x.txt"}}, TOOLS)

KeyError: 'save_file'

Fix: treat an unknown name as an error result that lists the real tools, as `run_tool` does.

In [19]:
print(run_tool({"name": "save_file", "args": {"filename": "x.txt"}}, TOOLS))

{'content': "unknown tool 'save_file'. Available tools: search_notes, calculator, write_file.", 'is_error': True}


## Try it
`calculator("2^3")` fails with "not allowed in a calculation: BinOp(...)", which does not help a model.
Write `calculator_v2` that catches that `ValueError` and raises a new one saying which operators are
allowed and how to write a power (`2*2*2`). Then write a fake script that calls `calculator` with
`"2^3"`, retries with `"2*2*2"` after an error, and answers from the result. Run it through `run_agent`
with `calculator_v2` as the `calculator` tool and check `errors == 1` and the answer contains 8.

In [20]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [21]:
llm_replay.stop()

## Key terms
- **error as observation** — a tool failure is caught and sent back to the model as the tool's result, so the model can see it and change its next call instead of the run crashing.
- **is_error flag** — a marker on a tool result saying the tool failed: `is_error: true` on an Anthropic `tool_result`, an `"error"` key in a Gemini `FunctionResponse.response`.
- **actionable error message** — an error text that says what was wrong and what a valid call looks like, e.g. "Retry with just a name, e.g. 'mars.txt'".
- **error result** — a tool result with the error flag set, carrying a short message (type and text, no stack trace).